# 온라인 기능(Online Features)
Feast는 온라인 저장소를 활용하여 추론 시간에 실시간으로 가져올 수 있는 온라인 기능을 저장합니다.  
우리는 `1-setup_feast.ipynb` 노트북에서 _materialize_를 실행했을 때 이 온라인 데이터베이스를 채웠습니다.  
온라인 저장소에서, 우리는 등록된 기능들에 대해서만 최신 기능 값을 저장합니다. 이것이 우리가 온라인 검색을 수행할 때 오프라인 검색 중에 했던 것과 달리 시간을 명시할 필요가 없는 이유입니다.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import feast
import pandas as pd
from datetime import datetime
import yaml

In [ ]:
with open('feature_repo/feature_store.yaml', 'r') as file:
    fs_config_yaml = yaml.safe_load(file)
fs_config = feast.repo_config.RepoConfig(**fs_config_yaml)
fs = feast.FeatureStore(config=fs_config)

여기서 우리는 (Feature View 대신) Feature Service를 사용합시다. 프로덕션 환경에서처럼 하는 것이 낫습니다.  
우리는 우리의 기능을 패키징하는 단일 기능 정의를 가리키는 것이 더 좋습니다.

In [ ]:
feature_service = fs.get_feature_service("serving_fs")

그리고 나서 우리는 노래의 ID를 얻고 Feast에 Feature Service에 정의된 기능 기반으로 최신 기능 값을 우리에게 달라고 요청합니다.

In [ ]:
song_properties = pd.read_parquet('../99-data_prep/song_properties.parquet')
favorite_song = song_properties.loc[song_properties["name"]=="Not Like Us"]
favorite_song

In [ ]:
online_features = fs.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "spotify_id": favorite_song["spotify_id"].values[0],
        }
    ],
)

In [ ]:
features = online_features.to_dict()
features

보시다시피, 이들은 우리가 이 워크숍 동안 잘 알게 된 속성들입니다. 이제 우리는 주어진 노래에 대해 최신 버전을 매우 쉽게 접근할 수 있는 방법이 있습니다! 🎶

## 추론(Inference)을 위해 이들을 사용하기
추론을 위해 이 값들을 사용하는 것은 우리가 이전에 한 것과 다르지 않습니다. 우리는 단순히 데이터로서 그들을 우리의 제공되는 모델에 전송하고 예측을 다시 얻습니다.

In [ ]:
import requests

In [ ]:
deployed_model_name = "jukebox"
infer_endpoint = "<paste-the-link-here>"
infer_url = f"{infer_endpoint}/v2/models/{deployed_model_name}/infer"

In [ ]:
def rest_request(data):
    json_data = {
        "inputs": [ 
           {
                "name": name,
                "shape": [1, 1],
                "datatype": "FP32",
                "data": [float(data[name][0])]
            }
            for name in data.keys()
        ]
    }

    response = requests.post(infer_url, json=json_data, verify=False)
    response_dict = response.json()
    return response_dict['outputs'][0]['data']

In [ ]:
data = features.copy()
del data["spotify_id"]
prediction = rest_request(data)
prediction

Now that we know how to easily get the relevant features for any song just through the ID, we can use this as a pre-processing step to make sure that even if the model or feature definition changes, we will still feed the right data into the model.